## Task 05 | Expression Validator and Evaluator

In [2]:
def evaluate_expression(expr):
    if type(expr) != str:
        raise TypeError("Expression string honi chahiye")

    # Step 1: Expression ko tokens (numbers aur operators) mein todein
    tokens = []
    i = 0
    n = len(expr)

    while i < n:
        ch = expr[i]

        # Whitespace ignore karein
        if ch.isspace():
            i += 1
            continue

        # Multi-digit number read karein
        if ch.isdigit():
            num_str = ""
            while i < n and expr[i].isdigit():
                num_str += expr[i]
                i += 1
            tokens.append(int(num_str))
            continue

        # Valid operators aur parentheses
        if ch in "+-*/()":
            tokens.append(ch)
            i += 1
            continue

        # Koi bhi invalid character aane par error dein
        raise ValueError("Invalid character mila: " + ch)

    if len(tokens) == 0:
        raise ValueError("Expression khali hai")

    # Helper function: operators ki priority check karne ke liye
    def precedence(op):
        if op in ("+", "-"):
            return 1
        if op in ("*", "/"):
            return 2
        return 0

    # Helper function: top operator ko top 2 numbers par apply karne ke liye
    def apply_operator(ops, values):
        if len(values) < 2:
            raise ValueError("Malformed expression: operands kam hain")
        op = ops.pop()
        right = values.pop()
        left = values.pop()

        if op == "+":
            values.append(left + right)
        elif op == "-":
            values.append(left - right)
        elif op == "*":
            values.append(left * right)
        elif op == "/":
            if right == 0:
                raise ZeroDivisionError("Division by zero allowed nahi hai")
            values.append(left / right)

    values = []
    ops = []
    expect_operand = True  # Track karta hai ke agla token number ya '(' hona chahiye

    for token in tokens:
        if type(token) == int:
            if not expect_operand:
                raise ValueError("Malformed expression: do numbers lagatar nahi aa sakte")
            values.append(float(token))
            expect_operand = False

        elif token == "(":
            if not expect_operand:
                raise ValueError("Malformed expression: '(' se pehle operator hona chahiye")
            ops.append(token)
            expect_operand = True

        elif token == ")":
            if expect_operand:
                raise ValueError("Malformed expression: ')' se pehle number hona chahiye")
            while len(ops) > 0 and ops[-1] != "(":
                apply_operator(ops, values)
            if len(ops) == 0:
                raise ValueError("Malformed expression: extra ')' mila")
            ops.pop()  # '(' ko stack se hatayein
            expect_operand = False

        else:
            # Token ek operator hai (+, -, *, /)
            if expect_operand:
                # Unary minus ya unary plus handle karein (jaise -5 ya -(3+2))
                if token == "-":
                    values.append(0.0)
                    ops.append("-")
                    continue
                elif token == "+":
                    continue
                else:
                    raise ValueError("Malformed expression: operator ghalat jagah par hai")

            while len(ops) > 0 and ops[-1] != "(" and precedence(ops[-1]) >= precedence(token):
                apply_operator(ops, values)

            ops.append(token)
            expect_operand = True

    if expect_operand:
        raise ValueError("Malformed expression: aakhir mein operator adhoora hai")

    # Baqi bache hue operators ko solve karein
    while len(ops) > 0:
        if ops[-1] == "(":
            raise ValueError("Malformed expression: '(' band nahi hua")
        apply_operator(ops, values)

    if len(values) != 1:
        raise ValueError("Malformed expression")

    return values[0]


# Testing
print(evaluate_expression("-(3 + 2) * 4 + 18 / 3"))  
print(evaluate_expression("10 + 2 * 6"))             
print(evaluate_expression("-5 * -2 + (8 - 3)"))      

-14.0
22.0
15.0
